In [ ]:
from pathlib import Path


def find_project_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in (current, *current.parents):
        if (path / "data").exists():
            return path
    return current


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data"

import csv
import json
import os
import random
import re
import threading
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional, Set

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import tushare as ts

TOKEN = os.getenv("TUSHARE_TOKEN", "").strip()
HTTP_URL = os.getenv("TUSHARE_HTTP_URL", "").strip()
STOCK_LIST_DIR = DATA_ROOT / "data_download" / "tushare_min_download" / "stock_list"
STOCK_DAY_DIR = DATA_ROOT / "data_download" / "tusahre_day_download" / "stock_day"
OUT_DIR = DATA_ROOT / "data_download" / "tushare_min_download" / "stock_minute"
START_DATE = "20170101"
END_DATE = "20270101"
ADJ = "hfq"
ADJ_FACTOR = True
FREQ = "1min"
ASSET = "E"
LIMIT = 8000
MAX_OFFSET = 100000
CHUNK_TRADE_DAYS = 30
MAX_RETRY = 25
BASE_SLEEP = 0.2
REQUEST_TIMEOUT_SEC = 20
ENABLE_OVERWRITE = False
ENABLE_JITTER = False
JITTER_RANGE_SEC = (0.05, 0.25)
JITTER_EVERY_CALL = True
ENABLE_PARALLEL = True
STOCK_WORKERS = 8
PROGRESS_EVERY = 1
RATE_LIMIT_PER_MIN = 230
RATE_PERIOD_SEC = 60
COMPRESSION = "snappy"

if not TOKEN:
    raise RuntimeError("TUSHARE_TOKEN is required")

pro = ts.pro_api(TOKEN)
if HTTP_URL:
    pro._DataApi__http_url = HTTP_URL
try:
    pro._DataApi__timeout = REQUEST_TIMEOUT_SEC
except Exception:
    pass


def jitter_sleep():
    if not ENABLE_JITTER:
        return
    low, high = JITTER_RANGE_SEC
    if high > 0:
        time.sleep(random.uniform(float(low), float(high)))


class CooldownGate:
    def __init__(self):
        self.lock = threading.Lock()
        self.next_allowed = 0.0

    def set_cooldown(self, seconds):
        with self.lock:
            self.next_allowed = max(self.next_allowed, time.time() + float(seconds))

    def wait(self):
        while True:
            with self.lock:
                next_allowed = self.next_allowed
            now = time.time()
            if now >= next_allowed:
                return
            time.sleep(min(1.0, next_allowed - now))


class RateLimiter:
    def __init__(self, max_calls, period_sec):
        self.max_calls = int(max_calls)
        self.period_sec = float(period_sec)
        self.lock = threading.Lock()
        self.times = deque()

    def acquire(self):
        while True:
            with self.lock:
                now = time.time()
                while self.times and now - self.times[0] >= self.period_sec:
                    self.times.popleft()
                if len(self.times) < self.max_calls:
                    self.times.append(now)
                    return
                wait_sec = self.period_sec - (now - self.times[0])
            time.sleep(max(0.05, min(1.0, wait_sec)))


gate = CooldownGate()
rate_limiter = RateLimiter(RATE_LIMIT_PER_MIN, RATE_PERIOD_SEC)


def retry_after_seconds(message):
    lower = message.lower()
    if "too many" in lower or "rate limit" in lower or "frequency" in lower:
        match = re.search(r"(\d+)", message)
        return int(match.group(1)) if match else 30
    if "connection reset" in lower or "connection aborted" in lower or "10054" in lower or "forcibly closed" in lower:
        return 20
    if "timeout" in lower:
        return 10
    return None


def call_with_rate_limit(fn, max_retry=MAX_RETRY, base_sleep=BASE_SLEEP, **kwargs):
    last_error = None
    for attempt in range(max_retry):
        gate.wait()
        rate_limiter.acquire()
        try:
            out = fn(**kwargs)
            if ENABLE_JITTER and JITTER_EVERY_CALL:
                jitter_sleep()
            return out
        except Exception as exc:
            last_error = exc
            wait_sec = retry_after_seconds(str(exc))
            if wait_sec is not None:
                gate.set_cooldown(wait_sec + random.uniform(0.2, 1.0))
                continue
            time.sleep(min(10.0, base_sleep * (2 ** attempt)) + random.uniform(0.0, 0.5))
    raise last_error


def list_trade_dates(start_date, end_date):
    dates = []
    if not STOCK_DAY_DIR.exists():
        return dates
    for path in STOCK_DAY_DIR.glob("*.parquet"):
        name = path.stem
        if len(name) == 8 and name.isdigit() and start_date <= name <= end_date:
            dates.append(name)
    return sorted(dates)


def load_codes_for_date(trade_date):
    path = STOCK_LIST_DIR / f"{trade_date}.json"
    if not path.exists():
        return []
    obj = json.loads(path.read_text(encoding="utf-8"))
    codes = obj["codes"] if isinstance(obj, dict) and "codes" in obj else obj
    return [str(code) for code in codes]


def build_universe_codes(trade_dates):
    values: Set[str] = set()
    missing = 0
    for trade_date in trade_dates:
        codes = load_codes_for_date(trade_date)
        if not codes:
            missing += 1
            continue
        values.update(codes)
    codes = sorted(values)
    print(f"universe days={len(trade_dates)} missing_json={missing} codes={len(codes)}")
    return codes


def detect_time_column(frame):
    for column in ["trade_time", "datetime", "time", "date", "trade_date"]:
        if column in frame.columns:
            return column
    raise ValueError(f"no time column columns={frame.columns.tolist()}")


def chunk_list(values, size):
    for start in range(0, len(values), size):
        yield values[start:start + size]


def safe_stock_filename(ts_code):
    return f"{ts_code}.parquet"


class StockParquetWriter:
    def __init__(self, out_path, compression="snappy"):
        self.out_path = out_path
        self.compression = compression
        self.writer: Optional[pq.ParquetWriter] = None
        self.columns: Optional[List[str]] = None

    def align(self, frame):
        if self.columns is None:
            self.columns = frame.columns.tolist()
            return frame
        for column in self.columns:
            if column not in frame.columns:
                frame[column] = pd.NA
        extra = [column for column in frame.columns if column not in self.columns]
        if extra:
            frame = frame.drop(columns=extra)
        return frame[self.columns]

    def write(self, frame):
        if frame is None or frame.empty:
            return
        frame = self.align(frame)
        table = pa.Table.from_pandas(frame, preserve_index=False)
        if self.writer is None:
            self.writer = pq.ParquetWriter(self.out_path, table.schema, compression=self.compression)
        self.writer.write_table(table)

    def close(self):
        if self.writer is not None:
            self.writer.close()
        self.writer = None


class YearCSVLogger:
    def __init__(self, year_dir, year):
        self.year_dir = year_dir
        self.year = year
        self.path = Path(year_dir) / f"{year}_missing_log.csv"
        self.lock = threading.Lock()
        Path(year_dir).mkdir(parents=True, exist_ok=True)
        if not self.path.exists():
            with self.path.open("w", newline="", encoding="utf-8") as f:
                csv.writer(f).writerow(["log_time", "year", "ts_code", "chunk_start", "chunk_end", "event", "missing_dates", "message"])

    def log(self, ts_code, chunk_start, chunk_end, event, missing_dates, message=""):
        row = [time.strftime("%Y-%m-%d %H:%M:%S"), self.year, ts_code, chunk_start, chunk_end, event, ";".join(missing_dates), message]
        with self.lock:
            with self.path.open("a", newline="", encoding="utf-8") as f:
                csv.writer(f).writerow(row)


def fetch_code_chunk(ts_code, start_date, end_date):
    parts = []
    offset = 0
    while True:
        frame = call_with_rate_limit(
            ts.pro_bar,
            ts_code=ts_code,
            api=pro,
            asset=ASSET,
            freq=FREQ,
            adj=ADJ,
            adjfactor=ADJ_FACTOR,
            start_date=start_date,
            end_date=end_date,
            limit=LIMIT,
            offset=offset,
        )
        if frame is None or frame.empty:
            break
        parts.append(frame)
        if len(frame) < LIMIT:
            break
        offset += LIMIT
        if offset >= MAX_OFFSET:
            break
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def normalize_and_filter(frame, ts_code, chunk_dates):
    if frame is None or frame.empty:
        return pd.DataFrame()
    if "ts_code" in frame.columns:
        frame = frame.rename(columns={"ts_code": "code"})
    else:
        frame["code"] = ts_code
    time_column = detect_time_column(frame)
    frame["datetime"] = pd.to_datetime(frame[time_column], errors="coerce")
    frame = frame[frame["datetime"].notna()]
    frame["date"] = frame["datetime"].dt.strftime("%Y%m%d")
    frame = frame[frame["date"].isin(chunk_dates)]
    if frame.empty:
        return pd.DataFrame()
    frame = frame.sort_values("datetime", ascending=True)
    frame = frame.drop_duplicates(subset=["datetime"], keep="last")
    first = ["code", "date", "datetime"]
    rest = [column for column in frame.columns if column not in first]
    return frame[first + rest]


def run_one_stock(ts_code, trade_dates, year_dir, year_logger):
    Path(year_dir).mkdir(parents=True, exist_ok=True)
    out_path = Path(year_dir) / safe_stock_filename(ts_code)
    if out_path.exists() and not ENABLE_OVERWRITE:
        return "skip"
    tmp_path = Path(str(out_path) + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()
    writer = StockParquetWriter(str(tmp_path), compression=COMPRESSION)
    wrote_any = False
    try:
        for chunk_dates in chunk_list(trade_dates, CHUNK_TRADE_DAYS):
            start_date = chunk_dates[0]
            end_date = chunk_dates[-1]
            chunk_set = set(chunk_dates)
            try:
                raw = fetch_code_chunk(ts_code, start_date, end_date)
            except Exception as exc:
                year_logger.log(ts_code, start_date, end_date, "FETCH_ERROR", chunk_dates, repr(exc))
                continue
            if raw is None or raw.empty:
                year_logger.log(ts_code, start_date, end_date, "RAW_EMPTY", chunk_dates, "raw is empty")
                continue
            try:
                frame = normalize_and_filter(raw, ts_code, chunk_set)
            except Exception as exc:
                year_logger.log(ts_code, start_date, end_date, "NORMALIZE_ERROR", chunk_dates, repr(exc))
                continue
            if frame is None or frame.empty:
                year_logger.log(ts_code, start_date, end_date, "FILTER_EMPTY", chunk_dates, "filtered data is empty")
                continue
            present_dates = set(frame["date"].astype(str).unique().tolist())
            missing_dates = sorted(chunk_set - present_dates)
            if missing_dates:
                year_logger.log(ts_code, start_date, end_date, "PARTIAL_MISSING", missing_dates, "chunk partially missing dates")
            writer.write(frame)
            wrote_any = True
            if ENABLE_JITTER and not JITTER_EVERY_CALL:
                jitter_sleep()
    finally:
        writer.close()
    if not wrote_any:
        if tmp_path.exists():
            tmp_path.unlink()
        return "empty"
    if out_path.exists():
        out_path.unlink()
    tmp_path.replace(out_path)
    return "ok"


def iter_year_ranges(start_date, end_date):
    years = range(int(start_date[:4]), int(end_date[:4]) + 1)
    ranges = []
    for year in years:
        start = max(start_date, f"{year}0101")
        end = min(end_date, f"{year}1231")
        if start <= end:
            ranges.append((start, end, year))
    return ranges


def run_one_year(start_date, end_date, year):
    trade_dates = [date for date in list_trade_dates(start_date, end_date) if date.startswith(str(year))]
    if not trade_dates:
        print(f"year={year} status=no_trade_dates")
        return
    codes = build_universe_codes(trade_dates)
    if not codes:
        print(f"year={year} status=no_codes")
        return
    year_dir = OUT_DIR / str(year)
    year_logger = YearCSVLogger(year_dir, year)
    total = len(codes)
    ok = empty = skip = fail = 0
    lock = threading.Lock()
    print(f"year={year} codes={total} days={len(trade_dates)} output={year_dir}")

    def run_code(code):
        nonlocal ok, empty, skip, fail
        try:
            result = run_one_stock(code, trade_dates, year_dir, year_logger)
            with lock:
                if result == "ok":
                    ok += 1
                elif result == "empty":
                    empty += 1
                elif result == "skip":
                    skip += 1
        except Exception as exc:
            with lock:
                fail += 1
                if fail <= 10:
                    print(f"fail year={year} code={code} error={repr(exc)}")

    if not ENABLE_PARALLEL or STOCK_WORKERS <= 1:
        for index, code in enumerate(codes, 1):
            run_code(code)
            if index % PROGRESS_EVERY == 0 or index == total:
                print(f"progress year={year} done={index}/{total} ok={ok} empty={empty} skip={skip} fail={fail}")
    else:
        with ThreadPoolExecutor(max_workers=STOCK_WORKERS) as executor:
            futures = [executor.submit(run_code, code) for code in codes]
            for index, _ in enumerate(as_completed(futures), 1):
                if index % PROGRESS_EVERY == 0 or index == total:
                    print(f"progress year={year} done={index}/{total} ok={ok} empty={empty} skip={skip} fail={fail}")
    print(f"year_done year={year} ok={ok} empty={empty} skip={skip} fail={fail}")


def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    ranges = iter_year_ranges(START_DATE, END_DATE)
    print(f"plan years={len(ranges)} start={START_DATE} end={END_DATE}")
    for start_date, end_date, year in ranges:
        run_one_year(start_date, end_date, year)
    print(f"done output={OUT_DIR}")


if __name__ == "__main__":
    main()